# Severity Index

Combines the impact variables into five pillars (population, housing, agriculture, livestock, infrastructure) at event level, takes the historical maximum per canton and province, normalizes each pillar (min-max) and weights them into a 0-1 severity index.

Outputs: `severity_index_*` tables in `data/processed/`.

## 0. Setup and weights

In [ ]:

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd
# Numerical operations
import numpy as np
# Efficient reader for GeoPackage layers
import pyogrio

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step4_severity_index", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("severidad iniciada — configuración cargada desde %s", CONFIG_PATH)

# Severity index settings from the config
SEV_CFG = config["severity_index"]
# Weight multiplier for destroyed assets versus affected ones
W_DESTROYED = SEV_CFG["destroyed_weight_multiplier"]
# Weight multiplier for deaths and missing persons
W_DEATH = SEV_CFG["death_weight_multiplier"]
# Weight multiplier for injured persons
W_INJURED = SEV_CFG["injured_weight_multiplier"]
# Weights of the five pillars
PILLAR_WEIGHTS = SEV_CFG["pillar_weights"]
# Check the pillar weights add up to 1
assert abs(sum(PILLAR_WEIGHTS.values()) - 1.0) < 1e-9, "los pesos de pilares deben sumar 1.0"
# Log the pillar weights
logger.info("Pesos de pilares: %s", PILLAR_WEIGHTS)

## 1. Load data

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Log the number of events loaded
logger.info("hazard_event_geo cargado: %d filas", len(he))

# Load the official canton and province names
adm2 = pyogrio.read_dataframe(
    processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False
)[["canton_code", "canton_name_official", "province_name_official"]].drop_duplicates()
# Remember the row count to detect accidental changes
n_antes = len(he)
# Attach official canton and province names to each event
he = he.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")
# Check the name join neither duplicates nor loses events
assert len(he) == n_antes, "el join adm2 no debe duplicar ni perder filas"
# Check every event has an official canton and province name
assert he[["canton_name_official", "province_name_official"]].isna().sum().sum() == 0, \
    "nombre oficial de cantón/provincia debe cubrir el 100% (canton_code es autoritativo y completo)"
# Log that official names were attached by canton code
logger.info("Nombres oficiales incorporados por atributo (canton_code); cobertura 100%%")

## 2. Pillar scores at event level

In [ ]:

# Population score: affected, evacuated and sheltered people plus weighted deaths, missing and injured
he["score_poblacion"] = (
    he["affected_direct"] + he["affected_indirect"] + he["evacuated"] + he["sheltered"]
    + W_DEATH * (he["deaths"] + he["missing_persons"])
    + W_INJURED * he["injured"]
)

# Housing score: affected houses plus weighted destroyed houses
he["score_vivienda"] = he["houses_affected"] + W_DESTROYED * he["houses_destroyed"]

# Agriculture score: affected hectares plus weighted destroyed hectares
he["score_agricultura"] = he["crops_affected_ha"] + W_DESTROYED * he["crops_destroyed_ha"]

# Livestock score: affected animals plus weighted deaths
he["score_ganaderia"] = he["livestock_affected"] + W_DESTROYED * he["livestock_deaths"]

# Infrastructure score: affected and destroyed schools and health facilities
he["score_infraestructura"] = (
    he["schools_affected"] + W_DESTROYED * he["schools_destroyed"]
    + he["health_facilities_affected"] + W_DESTROYED * he["health_facilities_destroyed"]
)

# Names of the five pillar score columns
PILLAR_SCORE_COLS = [
    "score_poblacion", "score_vivienda", "score_agricultura",
    "score_ganaderia", "score_infraestructura",
]
# Log that the pillar scores were computed
logger.info("Puntajes por pilar calculados a nivel evento (%d filas)", len(he))
# Loop over the pillar scores
for col in PILLAR_SCORE_COLS:
    # Check no score is negative
    assert (he[col] >= 0).all(), f"{col} tiene valores negativos — revisar datos de origen"

## 3. Aggregation: historical maximum per administrative unit

In [ ]:
# Aggregate pillar scores to the historical maximum per unit
def agregar_maximo(df: pd.DataFrame, group_cols: list[str], score_cols: list[str]) -> pd.DataFrame:
    """Take the historical maximum of each pillar score per administrative unit.
    
    Args:
        df: Event-level table with pillar scores.
        group_cols: Columns that define the unit.
        score_cols: Pillar score columns.
    
    Returns:
        Table with the maximum score per unit and the number of events.
    """
    # Group events by the requested administrative columns
    g = df.groupby(group_cols)
    # Maximum of each pillar per unit
    out = g[score_cols].max().reset_index()
    # Number of events per unit
    out["n_eventos"] = g.size().values
    # Return the per-unit table
    return out


# Severity aggregation for one administrative level
def ejecutar_nivel_severidad(df: pd.DataFrame, admin_cols: list[str]) -> pd.DataFrame:
    """Aggregate pillar maxima for one administrative level, per hazard and combined.
    
    Args:
        df: Event-level table with pillar scores.
        admin_cols: Columns that define the administrative level.
    
    Returns:
        Table stacking the per-hazard rows and the combined ('TODOS') rows.
    """
    # Maximum per unit and hazard type
    por_hazard = agregar_maximo(df, admin_cols + ["hazard_type"], PILLAR_SCORE_COLS)

    # Maximum per unit across all hazards
    combinado = agregar_maximo(df, admin_cols, PILLAR_SCORE_COLS)
    # Re-apply the combined label after the join
    combinado["hazard_type"] = "TODOS"

    # Stack the per-hazard and combined views
    return pd.concat([por_hazard, combinado], ignore_index=True)

In [ ]:
# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]
# Columns that identify a province
PROVINCE_COLS = ["province_code", "province_name_official"]

# Log the start of the canton-level aggregation
logger.info("Agregando máximo histórico a nivel cantón...")
# Aggregate at canton level
sev_canton = ejecutar_nivel_severidad(he, CANTON_COLS)

# Log the start of the province-level aggregation
logger.info("Agregando máximo histórico a nivel provincia...")
# Aggregate at province level
sev_provincia = ejecutar_nivel_severidad(he, PROVINCE_COLS)

# Log the number of rows produced
logger.info("Cantón: %d filas | Provincia: %d filas", len(sev_canton), len(sev_provincia))

## 4. Min-max normalization per pillar

In [ ]:
# Min-max normalization of each pillar within each hazard type
def normalizar_minmax(df: pd.DataFrame, score_cols: list[str], by: str = "hazard_type") -> pd.DataFrame:
    """Scale each pillar score to 0-1 within each group.
    
    Args:
        df: Table with pillar score columns.
        score_cols: Score columns to normalize.
        by: Column that defines the groups (default: hazard type).
    
    Returns:
        Copy of the table with 'norm_' columns added.
    """
    # Work on a copy
    df = df.copy()
    # Loop over the pillars
    for col in score_cols:
        # Name of the normalized column
        norm_col = col.replace("score_", "norm_")
        # Min-max scaling of one series
        def _minmax(s: pd.Series) -> pd.Series:
            # Range of the series
            rango = s.max() - s.min()
            # A constant series has no range
            if rango == 0:
                # Return zeros for a constant series
                return pd.Series(0.0, index=s.index)
            # Scale to the 0-1 interval
            return (s - s.min()) / rango
        # Apply the scaling within each hazard group
        df[norm_col] = df.groupby(by)[col].transform(_minmax)
    # Return the normalized table
    return df

In [ ]:
# Normalize canton pillars
sev_canton = normalizar_minmax(sev_canton, PILLAR_SCORE_COLS)
# Normalize province pillars
sev_provincia = normalizar_minmax(sev_provincia, PILLAR_SCORE_COLS)

# Names of the normalized columns
NORM_COLS = [c.replace("score_", "norm_") for c in PILLAR_SCORE_COLS]
# Loop over both levels to validate them
for df_, nombre in [(sev_canton, "cantón"), (sev_provincia, "provincia")]:
    # Loop over the normalized columns
    for col in NORM_COLS:
        # Check every normalized value is between 0 and 1
        assert df_[col].between(0, 1).all(), f"{col} fuera de [0,1] en la vista {nombre}"
# Log that the normalization is valid
logger.info("Normalización min-max aplicada y verificada en [0,1]")

## 5. Weighted severity index

In [ ]:
# Weighted severity index from the normalized pillars
def calcular_severity_index(df: pd.DataFrame, weights: dict[str, float]) -> pd.DataFrame:
    """Compute the weighted severity index from the normalized pillars.
    
    Args:
        df: Table with the five normalized pillar columns.
        weights: Weight per pillar; must add up to 1.
    
    Returns:
        Copy of the table with the 'severity_index' column in [0, 1].
    """
    # Work on a copy
    df = df.copy()
    # Weighted sum of the five normalized pillars
    df["severity_index"] = (
        weights["poblacion"] * df["norm_poblacion"]
        + weights["vivienda"] * df["norm_vivienda"]
        + weights["agricultura"] * df["norm_agricultura"]
        + weights["ganaderia"] * df["norm_ganaderia"]
        + weights["infraestructura"] * df["norm_infraestructura"]
    )
    # Return the table with the index
    return df

In [ ]:
# Severity index at canton level
sev_canton = calcular_severity_index(sev_canton, PILLAR_WEIGHTS)
# Severity index at province level
sev_provincia = calcular_severity_index(sev_provincia, PILLAR_WEIGHTS)

# Check canton values are between 0 and 1
assert sev_canton["severity_index"].between(0, 1).all(), "severity_index fuera de [0,1] (cantón)"
# Check province values are between 0 and 1
assert sev_provincia["severity_index"].between(0, 1).all(), "severity_index fuera de [0,1] (provincia)"
# Log that the index is valid at both levels
logger.info("Índice de Severidad calculado y verificado en [0,1] para ambos niveles")

## 6. Quantile classification (provisional, not the final composite classification)

In [ ]:
# Names of the five classes
CLASES = ["Muy Baja", "Baja", "Moderada", "Alta", "Muy Alta"]

# Classify values into quantile classes
def clasificar_cuantiles(serie: pd.Series, clases: list[str]) -> pd.Series:
    """Classify values into quantile classes.
    
    Args:
        serie: Values to classify.
        clases: Class labels, from lowest to highest.
    
    Returns:
        The class of each value; falls back to a rank-based cut when ties break the quantiles.
    """
    # Try the standard quantile cut
    try:
        # Cut into five equal-count classes
        return pd.qcut(serie, q=len(clases), labels=clases, duplicates="drop")
    except ValueError:
        # Log the fallback when ties break the cut
        logger.warning("qcut con %d clases falló por empates; usando rank-based fallback", len(clases))
        # Rank-based cut that handles ties
        return pd.qcut(serie.rank(method="first"), q=len(clases), labels=clases)


# Select the combined hazard view at canton level
mask_todos_canton = sev_canton["hazard_type"] == "TODOS"
# Classify canton severity
sev_canton.loc[mask_todos_canton, "clase_severidad"] = clasificar_cuantiles(
    sev_canton.loc[mask_todos_canton, "severity_index"], CLASES
)

# Select the combined hazard view at province level
mask_todos_prov = sev_provincia["hazard_type"] == "TODOS"
# Classify province severity
sev_provincia.loc[mask_todos_prov, "clase_severidad"] = clasificar_cuantiles(
    sev_provincia.loc[mask_todos_prov, "severity_index"], CLASES
)

# Show the class distribution at canton level
print(sev_canton.loc[mask_todos_canton, "clase_severidad"].value_counts())

## 7. Validation

In [ ]:
# Number of cantons with historical events
n_cantones_esperados = he[CANTON_COLS].drop_duplicates().shape[0]
# Number of cantons in the severity table
n_cantones_en_severidad = (sev_canton["hazard_type"] == "TODOS").sum()
# Log expected against present cantons
logger.info("Cantones con eventos: %d | cantones en severidad (TODOS): %d", n_cantones_esperados, n_cantones_en_severidad)
# Check no canton is missing or extra
assert n_cantones_esperados == n_cantones_en_severidad, "faltan o sobran cantones en la tabla de severidad"

# Number of provinces with historical events
n_provincias_esperadas = he[PROVINCE_COLS].drop_duplicates().shape[0]
# Number of provinces in the severity table
n_provincias_en_severidad = (sev_provincia["hazard_type"] == "TODOS").sum()
# Check no province is missing or extra
assert n_provincias_esperadas == n_provincias_en_severidad, "faltan o sobran provincias en la tabla de severidad"

# Check there are no missing values at canton level
assert sev_canton["severity_index"].isna().sum() == 0, "severity_index tiene nulos (cantón)"
# Check there are no missing values at province level
assert sev_provincia["severity_index"].isna().sum() == 0, "severity_index tiene nulos (provincia)"

# Ten cantons with the highest severity
top10 = sev_canton[mask_todos_canton].sort_values("severity_index", ascending=False).head(10)
# Print the top ten
print(top10[["province_name_official", "canton_name_official", "severity_index",
             "clase_severidad", "n_eventos"]].to_string(index=False))

## 8. Export

In [ ]:
# Tables to export
tablas = {
    "severity_index_canton": sev_canton,
    "severity_index_provincia": sev_provincia,
}

# Loop over the tables
for nombre, df in tablas.items():
    # Save as Parquet
    df.to_parquet(processed_dir / f"{nombre}.parquet", index=False)
    # Save as CSV
    df.to_csv(processed_dir / f"{nombre}.csv", index=False)
    # Log each exported table
    logger.info("Exportado: %s (%d filas)", nombre, len(df))

# Print the export folder
print(f"Tablas exportadas a {processed_dir} :")
# Loop over the exported tables
for nombre in tablas:
    # Print each table name
    print(f" - {nombre}")